# Malaysia Tax RAG — RAGAS + Policy Evaluation

In [1]:
import sys
import json
import re
import os
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path("../.env"))  # repo root

# Ensure notebooks dir is on path (same as generate_testset.ipynb)
_notebooks = Path(".").resolve()
if (_notebooks / "ingest_utils.py").exists():
    sys.path.insert(0, str(_notebooks))
else:
    sys.path.insert(0, str(Path("frontend/notebooks").resolve()))

# API base — change if server runs on a different port
API_BASE = os.getenv("EVAL_API_BASE", "http://localhost:8000")
GOLDEN_DATASET_PATH = Path(".").resolve() / "golden_dataset.json"
TOP_K = 4  # cost control: limit retrieved chunks

print(f"API_BASE: {API_BASE}")
print(f"Golden dataset: {GOLDEN_DATASET_PATH}")

API_BASE: http://localhost:8000
Golden dataset: /Users/victoria/Documents/AIBootcamp/The-AI-Engineer-Challenge/notebooks/golden_dataset.json


## Load Golden Dataset

In [2]:
with open(GOLDEN_DATASET_PATH) as f:
    golden = json.load(f)

df = pd.DataFrame(golden)
print(f"Loaded {len(df)} test cases")
print(f"Types: {df['type'].value_counts().to_dict()}")
try:
    display(df[["id", "question", "type"]].head(10))
except NameError:
    print(df[["id", "question", "type"]].head(10).to_string())

Loaded 30 test cases
Types: {'INFO': 26, 'WHAT_IF': 4}


,id,question,type
0,res_01,I spent 190 days in Malaysia in 2025. Am I res...,INFO
1,res_02,I was in Malaysia for exactly 182 days in 2025...,INFO
2,res_03,I was in Malaysia Oct 2025 to March 2026. Does...,INFO
3,res_04,I left Malaysia on 1 December 2025 and returne...,INFO
4,res_05,What is the 182-day rule for Malaysian tax res...,INFO
5,res_06,Can I be a tax resident in Malaysia if I was o...,WHAT_IF
6,res_07,I am a Malaysian citizen but I've lived abroad...,INFO
7,res_08,What happens if I am not a tax resident in Mal...,INFO
8,inc_01,I work remotely for a US company and am a Mala...,INFO
9,inc_02,I visited Malaysia for 45 days total and worke...,INFO


## Run Golden Dataset Against API (Eval Mode)

In [3]:
import asyncio

async def call_eval_api(question: str, client: httpx.AsyncClient) -> dict:
    """Call /app/chat/eval and return structured response."""
    try:
        resp = await client.post(
            f"{API_BASE}/app/chat/eval",
            headers={"X-Eval-Mode": "1", "Content-Type": "application/json"},
            json={"input": question},
            timeout=60.0,
        )
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"  ERROR for '{question[:50]}': {e}")
        return {"answer": "", "contexts": [], "policy": {}}


async def run_all(test_cases: list) -> list:
    results = []
    async with httpx.AsyncClient() as client:
        for tc in test_cases:
            print(f"  [{tc['id']}] {tc['question'][:60]}...")
            result = await call_eval_api(tc["question"], client)
            results.append({
                "id": tc["id"],
                "question": tc["question"],
                "ground_truth": tc.get("ground_truth_short", ""),
                "answer": result.get("answer", ""),
                # RAGAS needs contexts as list of strings
                "contexts": [c["text"] for c in result.get("contexts", [])[:TOP_K]],
                # Keep full context objects for analysis
                "contexts_full": result.get("contexts", [])[:TOP_K],
                "policy": result.get("policy", {}),
                "type": tc["type"],
                "requires_citation": tc["requires_citation"],
            })
    return results

print("Running golden dataset against API...")
print("(Ensure server is running at", API_BASE, ")")
eval_results = await run_all(golden)
print(f"\nCollected {len(eval_results)} responses")

Running golden dataset against API...
(Ensure server is running at http://localhost:8000 )
  [res_01] I spent 190 days in Malaysia in 2025. Am I resident for tax?...
  [res_02] I was in Malaysia for exactly 182 days in 2025. Am I a tax r...
  [res_03] I was in Malaysia Oct 2025 to March 2026. Does that make me ...
  [res_04] I left Malaysia on 1 December 2025 and returned on 15 Januar...
  [res_05] What is the 182-day rule for Malaysian tax residency?...
  [res_06] Can I be a tax resident in Malaysia if I was only here for 1...
  [res_07] I am a Malaysian citizen but I've lived abroad for 5 years. ...
  [res_08] What happens if I am not a tax resident in Malaysia?...
  [inc_01] I work remotely for a US company and am a Malaysian tax resi...
  [inc_02] I visited Malaysia for 45 days total and worked remotely for...
  [inc_03] I am a freelance contractor invoicing a Malaysian company. I...
  [inc_04] What income is considered Malaysia-sourced under the Income ...
  [inc_05] I receive a s

## Custom Policy Metrics — Deterministic (No LLM)

In [4]:
# ── Citation keywords (from evals.md) ─────────────────────────────────────────
_CITATION_NUMBERS_RE = re.compile(r"\b(60|182|183|30%)\b")
_CITATION_WORDS_RE = re.compile(
    r"\b(Section|PR|Article|Schedule|Form\s+BE|Form\s+B\b|Form\s+M\b)\b", re.IGNORECASE
)
_CITATION_MARKER_RE = re.compile(
    r"\b(s\d+|Section\s+\d+|PR\s+\d+/\d+|PR\s+\d+|Article\s+\d+|Schedule\s+\d+|Form\s+[BEM]E?|ITA)\b",
    re.IGNORECASE,
)

# Advice leakage phrases (from evals.md)
_ADVICE_LEAK_RE = re.compile(
    r"\b(you should|you must|definitely|you don't need|I recommend)\b", re.IGNORECASE
)

def one_question_compliance(answer: str) -> bool:
    """Pass if at most 1 question mark (excluding citation-style brackets)."""
    # Strip citation patterns like [Section 7] before counting
    cleaned = re.sub(r"\[.*?\]", "", answer)
    q_count = cleaned.count("?")
    return q_count <= 1

def citation_coverage(answer: str, requires_citation: bool) -> bool:
    """If answer contains citation-triggering content, it must have a citation marker."""
    if not requires_citation:
        return True  # not required — always pass
    has_trigger = bool(_CITATION_NUMBERS_RE.search(answer) or _CITATION_WORDS_RE.search(answer))
    if not has_trigger:
        return True  # no trigger found — pass (nothing to cite)
    return bool(_CITATION_MARKER_RE.search(answer))

def advice_leakage(answer: str) -> bool:
    """Return True (pass) if no advice leakage phrases found."""
    return not bool(_ADVICE_LEAK_RE.search(answer))


# ── Apply metrics ──────────────────────────────────────────────────────────────
policy_rows = []
for r in eval_results:
    ans = r["answer"]
    oqc = one_question_compliance(ans)
    cc = citation_coverage(ans, r["requires_citation"])
    al = advice_leakage(ans)
    policy_rows.append({
        "id": r["id"],
        "one_question_compliance": oqc,
        "citation_coverage": cc,
        "advice_leakage_pass": al,
        "num_q_marks": ans.count("?"),
        "num_contexts": len(r["contexts"]),
        "max_score": max((c.get("score", 0) for c in r["contexts_full"]), default=0.0),
    })

policy_df = pd.DataFrame(policy_rows)

oqc_pct = policy_df["one_question_compliance"].mean() * 100
cc_pct = policy_df["citation_coverage"].mean() * 100
al_pct = policy_df["advice_leakage_pass"].mean() * 100
avg_chunks = policy_df["num_contexts"].mean()
avg_max_score = policy_df["max_score"].mean()

print(f"One Question Compliance:  {oqc_pct:.1f}%")
print(f"Citation Coverage:        {cc_pct:.1f}%")
print(f"Advice Leakage (pass%):   {al_pct:.1f}%")
print(f"Avg chunks per question:  {avg_chunks:.2f}")
print(f"Avg max similarity score: {avg_max_score:.4f}")

try:
    display(policy_df)
except NameError:
    print(policy_df.to_string())

One Question Compliance:  96.7%
Citation Coverage:        86.7%
Advice Leakage (pass%):   100.0%
Avg chunks per question:  3.60
Avg max similarity score: 0.6627


,id,one_question_compliance,citation_coverage,advice_leakage_pass,num_q_marks,num_contexts,max_score
0,res_01,True,True,True,1,4,0.755115
1,res_02,True,True,True,1,4,0.734494
2,res_03,True,True,True,1,4,0.750693
3,res_04,True,True,True,1,4,0.719831
4,res_05,True,False,True,1,4,0.750257
5,res_06,True,True,True,1,4,0.768835
6,res_07,True,True,True,1,4,0.781814
7,res_08,True,True,True,1,4,0.760688
8,inc_01,True,True,True,1,4,0.739608
9,inc_02,True,True,True,1,4,0.782788


## RAGAS Evaluation (LLM-Assisted)

In [5]:
import warnings
import instructor
from datasets import Dataset
from ragas import evaluate
from ragas.dataset_schema import EvaluationResult
# ragas.metrics (not .collections) — these inherit from Metric, which evaluate() requires
from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerRelevancy
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings as LCOpenAIEmbeddings
from ragas.llms import LiteLLMStructuredLLM

# Filter to cases that have answers and contexts
ragas_rows = [
    r for r in eval_results
    if r["answer"] and r["contexts"]
]
print(f"Cases with answers+contexts for RAGAS: {len(ragas_rows)}")

ragas_data = {
    "question":      [r["question"] for r in ragas_rows],
    "answer":        [r["answer"] for r in ragas_rows],
    "contexts":      [r["contexts"] for r in ragas_rows],
    "ground_truth":  [r["ground_truth"] for r in ragas_rows],
}
ragas_dataset = Dataset.from_dict(ragas_data)

# LiteLLMStructuredLLM uses instructor for structured output — client must be patched
_oai_instructor = instructor.from_openai(OpenAI())
evaluator_llm = LiteLLMStructuredLLM(client=_oai_instructor, model="gpt-4o-mini", provider="openai")
# AnswerRelevancy (old ragas.metrics API) calls embed_query/embed_documents — needs LangChain embeddings
evaluator_embeddings = LCOpenAIEmbeddings(model="text-embedding-3-small")

# Instantiate then inject llm/embeddings as attributes (ragas.metrics style)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=DeprecationWarning)
    faithfulness_m   = Faithfulness();      faithfulness_m.llm   = evaluator_llm
    context_prec_m   = ContextPrecision();  context_prec_m.llm   = evaluator_llm
    context_rec_m    = ContextRecall();     context_rec_m.llm    = evaluator_llm
    # strictness=1: ask for 1 generated question instead of 3 — avoids "1 instead of 3" warning
    answer_rel_m     = AnswerRelevancy(strictness=1); answer_rel_m.llm = evaluator_llm
    answer_rel_m.embeddings = evaluator_embeddings
    metrics = [faithfulness_m, context_prec_m, context_rec_m, answer_rel_m]

ragas_result = evaluate(ragas_dataset, metrics=metrics)
assert isinstance(ragas_result, EvaluationResult), f"Unexpected return type: {type(ragas_result)}"

ragas_scores: pd.DataFrame = ragas_result.to_pandas()
print(f"\nRagas per-case scores (columns: {ragas_scores.columns.tolist()})")
metric_cols = [c for c in ["faithfulness", "context_precision", "context_recall", "answer_relevancy"] if c in ragas_scores.columns]
try:
    display(ragas_scores[["user_input"] + metric_cols])
except NameError:
    print(ragas_scores[["user_input"] + metric_cols].to_string())

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/l9/k4g359vd6vg5k697lr_4l69c0000gn/T/ipykernel_92962/3613035521.py:7: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerRelevancy
/var/folders/l9/k4g359vd6vg5k697lr_4l69c0000gn/T/ipykernel_92962/3613035521.py:7: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, ContextPrecision, Co

Cases with answers+contexts for RAGAS: 27


Evaluating: 100%|██████████| 108/108 [18:32<00:00, 10.30s/it] 



Ragas per-case scores (columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'context_precision', 'context_recall', 'answer_relevancy'])


,user_input,faithfulness,context_precision,context_recall,answer_relevancy
0,I spent 190 days in Malaysia in 2025. Am I res...,0.538462,1.000000,1.000000,0.803290
1,I was in Malaysia for exactly 182 days in 2025...,0.142857,1.000000,1.000000,0.767703
2,I was in Malaysia Oct 2025 to March 2026. Does...,0.466667,1.000000,0.500000,0.000000
3,I left Malaysia on 1 December 2025 and returne...,0.312500,0.833333,0.000000,0.000000
4,What is the 182-day rule for Malaysian tax res...,0.923077,0.750000,0.000000,0.000000
5,Can I be a tax resident in Malaysia if I was o...,0.818182,1.000000,0.333333,0.000000
6,I am a Malaysian citizen but I've lived abroad...,0.750000,1.000000,1.000000,0.797910
7,What happens if I am not a tax resident in Mal...,0.583333,0.916667,0.500000,0.797307
8,I work remotely for a US company and am a Mala...,0.636364,1.000000,0.666667,0.920310
9,I visited Malaysia for 45 days total and worke...,0.785714,1.000000,0.666667,0.000000


## Results Table

In [6]:
def safe_mean(series):
    try:
        return series.dropna().mean()
    except Exception:
        return float("nan")

faithfulness_score = safe_mean(ragas_scores["faithfulness"])       if "faithfulness"       in ragas_scores.columns else float("nan")
cp_score           = safe_mean(ragas_scores["context_precision"])  if "context_precision"  in ragas_scores.columns else float("nan")
cr_score           = safe_mean(ragas_scores["context_recall"])     if "context_recall"     in ragas_scores.columns else float("nan")
ar_score           = safe_mean(ragas_scores["answer_relevancy"])   if "answer_relevancy"   in ragas_scores.columns else float("nan")

results_table = pd.DataFrame([
    {"Metric": "Faithfulness",             "Score": f"{faithfulness_score:.2f}"},
    {"Metric": "Context Precision",        "Score": f"{cp_score:.2f}"},
    {"Metric": "Context Recall",           "Score": f"{cr_score:.2f}"},
    {"Metric": "Answer Relevancy",         "Score": f"{ar_score:.2f}"},
    {"Metric": "One Question Compliance",  "Score": f"{oqc_pct:.1f}%"},
    {"Metric": "Citation Coverage",        "Score": f"{cc_pct:.1f}%"},
    {"Metric": "Advice Leakage (pass%)",   "Score": f"{al_pct:.1f}%"},
])

print("=" * 40)
print("EVALUATION RESULTS SUMMARY")
print("=" * 40)
print(f"Total cases evaluated:        {len(eval_results)}")
print(f"Cases with RAGAS scores:      {len(ragas_rows)}")
print(f"Avg retrieved chunks/Q:       {avg_chunks:.2f}")
print(f"Avg max similarity score:     {avg_max_score:.4f}")
print()
try:
    display(results_table.set_index("Metric"))
except NameError:
    print(results_table.to_string(index=False))

# Diagnostic summary
print("\n── Diagnostic Notes ──")
low_oqc = policy_df[~policy_df["one_question_compliance"]]
if not low_oqc.empty:
    print(f"OQC failures ({len(low_oqc)}): {low_oqc['id'].tolist()}")
else:
    print("OQC: all passed")

low_cc = policy_df[~policy_df["citation_coverage"]]
if not low_cc.empty:
    print(f"Citation coverage failures ({len(low_cc)}): {low_cc['id'].tolist()}")
else:
    print("Citation coverage: all passed")

low_al = policy_df[~policy_df["advice_leakage_pass"]]
if not low_al.empty:
    print(f"Advice leakage detected ({len(low_al)}): {low_al['id'].tolist()}")
else:
    print("Advice leakage: none detected")

if "faithfulness" in ragas_scores.columns:
    low_faith_cases = ragas_scores[ragas_scores["faithfulness"] < 0.5]
    if not low_faith_cases.empty:
        print(f"Low faithfulness (<0.5): {[s[:50] for s in low_faith_cases['user_input'].astype(str).tolist()]}")

EVALUATION RESULTS SUMMARY
Total cases evaluated:        30
Cases with RAGAS scores:      27
Avg retrieved chunks/Q:       3.60
Avg max similarity score:     0.6627



,Score
Metric,
Faithfulness,0.70
Context Precision,0.88
Context Recall,0.55
Answer Relevancy,0.45
One Question Compliance,96.7%
Citation Coverage,86.7%
Advice Leakage (pass%),100.0%



── Diagnostic Notes ──
OQC failures (1): ['fil_04']
Citation coverage failures (4): ['res_05', 'inc_07', 'dta_01', 'dta_04']
Advice leakage: none detected
Low faithfulness (<0.5): ['I was in Malaysia for exactly 182 days in 2025. Am', 'I was in Malaysia Oct 2025 to March 2026. Does tha', 'I left Malaysia on 1 December 2025 and returned on', 'If I earn both employment and business income, how', 'What is a permanent establishment in Malaysian tax']


## Save Results

In [7]:
out_dir = Path(".").resolve()

# Save eval_results.json (full per-case results)
eval_output = []
for i, r in enumerate(eval_results):
    row = {
        "id": r["id"],
        "question": r["question"],
        "answer": r["answer"],
        "contexts": r["contexts"],
        "ground_truth": r["ground_truth"],
        "policy": r["policy"],
        "type": r["type"],
    }
    # Add RAGAS scores if available
    if i < len(ragas_scores):
        for col in ["faithfulness", "context_precision", "context_recall", "answer_relevancy"]:
            val = ragas_scores.iloc[i].get(col)
            if val is not None and not pd.isna(val):
                row[col] = float(val)
    eval_output.append(row)

eval_path = out_dir / "eval_results.json"
with open(eval_path, "w") as f:
    json.dump(eval_output, f, indent=2)
print(f"Saved eval_results.json → {eval_path}")

# Print final summary for record
print("\nFinal scores saved. Run complete.")

Saved eval_results.json → /Users/victoria/Documents/AIBootcamp/The-AI-Engineer-Challenge/notebooks/eval_results.json

Final scores saved. Run complete.
